# The Markov state

The answers to Section 9 of the sample questions, `documentation/tex/questions`, run.
The questions concern Section 1.3.3 of the lecture notes: the decayed counts $Z$ of an
exponential-kernel Hawkes process, Theorem 1.3.14 and the time homogeneity that follows it,
and their use in the forward means of Proposition 1.3.33 and in the update (1.3.38).
The two types are read as the market buys (type 0) and the market sells (type 1).

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
A Monte Carlo choice is decided by the score of a sample mean against its standard error:
a mean within four standard errors of its target is on it, and one beyond eight is off it.

In [ ]:
import contextlib
import io
import timeit

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook

from unito26.lob.hawkes import Clock, decayed_counts_at
from unito26.lob.monte_carlo import Fan, ForwardCounts, SampleMean


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")


def scores(sample, target):
    """The score of the mean of the rows of `sample` against `target`, a component each."""
    return SampleMean.from_sample(np.asarray(sample, dtype=float)).z_scores(target)


def on(score):
    return bool(np.all(np.abs(score) < 4))


def off(score):
    return bool(np.any(np.abs(score) > 8))

## `question.restartTheStateAfterAMarketSell`: the state continued from an event

With $\beta = \log 2$ the state halves in a unit of time.
On the whole history $Z(3) = (0.25, 0.5)$ and $Z(5) = (0.5625, 0.375)$.

In [ ]:
def run():
    beta = np.log(2.0)
    times = np.array([1.0, 2.0, 3.0, 4.0])
    types = np.array([0, 1, 1, 0])
    clock = Clock(np.array([3.0, 5.0]))
    zero = np.zeros(2)
    k = 1
    s = times[k]
    whole = decayed_counts_at(beta, times, types, zero, clock)
    before = decayed_counts_at(beta, times, types, zero, Clock([s]))[0]
    after = before + np.eye(2)[types[k]]

    def restart(z, keep):
        return decayed_counts_at(
            beta, times[keep] - s, types[keep], z, Clock(clock - s)
        )

    for z, keep in ((before, times >= s), (before, times > s),
                    (after, times > s), (after, times >= s)):
        print(np.allclose(restart(z, keep), whole))


whole = decayed_counts_at(
    np.log(2.0), np.array([1.0, 2.0, 3.0, 4.0]), np.array([0, 1, 1, 0]), np.zeros(2),
    Clock(np.array([3.0, 5.0])),
)
print("the whole history:", whole.tolist())
assert np.allclose(whole, [[0.25, 0.5], [0.5625, 0.375]])
printed = outcome(run).split("\n")
print(printed)
verify("question.restartTheStateAfterAMarketSell", "C", [
    printed == ["False", "False", "True", "False"],
    printed == ["False", "True", "False", "False"],
    printed == ["True", "False", "True", "False"],
    printed == ["True", "True", "True", "True"],
])

## `question.millisecondClockKeepsTheState`: the state and the forecast in milliseconds

Remark 1.3.11 with $c = 1000$: the rates are divided by $c$, the state is unchanged, and a
horizon is measured on the new clock.

In [ ]:
def run():
    seconds = config.one_way_pair_params()
    millis = HawkesParams(
        baseline=seconds.baseline / 1000,
        excitation=seconds.excitation / 1000,
        decay=seconds.decay / 1000,
    )
    times = np.array([0.2, 0.35, 0.9])
    types = np.array([0, 1, 1])
    zero = np.zeros(2)
    z = decayed_counts_at(seconds.decay, times, types, zero, Clock([1.0]))[0]
    z_ms = decayed_counts_at(
        millis.decay, 1000 * times, types, zero, Clock([1000.0])
    )[0]
    expected = seconds.forward_mean_counts(z, 0.5)
    print(np.allclose(z_ms, z / 1000), np.allclose(z_ms, z))
    print(np.allclose(millis.forward_mean_counts(z_ms, 0.5), expected),
          np.allclose(millis.forward_mean_counts(z_ms, 500.0), expected))


first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.millisecondClockKeepsTheState", "A", [
    first == "False True" and second == "False True",
    first == "True False" and second == "False True",
    first == "False True" and second == "True False",
    first == "False True" and second == "False False",
])

## `question.totalIntensityIsNotTheState`: two states of equal total intensity

Over ten seconds the progeny of the state is complete to within $e^{-10}$, and the gap of
the expected counts is $\sum_e (z_{1,e} - z_{2,e})(C_e - 1)$, with $C_e - 1$ the column sums
of $\Gamma(I - \Gamma)^{-1}$: 3 and 3 on the self-exciting pair, 3.375 and 1.5 on the
one-way pair.

In [ ]:
def run():
    def total_intensity(params, z):
        return (params.baseline + params.excitation @ z).sum()

    def total_count(params, z):
        return params.forward_mean_counts(z, 10.0).sum()

    for params, z1, z2 in (
        (config.self_exciting_pair_params(), [1.0, 0.0], [0.5, 0.5]),
        (config.one_way_pair_params(), [0.5, 0.0], [0.0, 1.0]),
    ):
        z1, z2 = np.array(z1), np.array(z2)
        gap = total_intensity(params, z1) - total_intensity(params, z2)
        more = total_count(params, z1) - total_count(params, z2)
        print(f"{gap:.2f} {more:.2f}")


for params in (config.self_exciting_pair_params(), config.one_way_pair_params()):
    gamma = params.branching_matrix
    print("C - 1:", (gamma @ np.linalg.inv(np.eye(2) - gamma)).sum(axis=0))
first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.totalIntensityIsNotTheState", "B", [
    first == "0.00 0.00" and second == "0.00 0.00",
    first == "0.00 0.00" and second == "0.00 0.19",
    first == "0.00 0.00" and second == "0.00 -0.19",
    first == "0.00 0.00" and second == "0.00 -0.31",
])

## `question.fanStartsTheClockAtZero`: four estimates of the expected counts after $s$

Each estimate is the mean of 3000 independent paths, scored against
`params.forward_mean_counts(z, h)`.

In [ ]:
params = config.one_way_pair_params()
z, s, h = np.array([0.5, 2.0]), 30.0, 1.0

def counts(simulator, start):
    ahead = [e for t, e in simulator.events(start + h) if t > start]
    return np.bincount(ahead, minlength=2)

def estimate_a(rng):
    return counts(ExponentialHawkes.from_state(params, z, rng), 0.0)

def estimate_b(rng):
    return counts(ExponentialHawkes.from_state(params, z, rng), s)

def estimate_c(rng):
    return counts(ExponentialHawkes(params, rng), s)

def estimate_d(rng):
    simulator = ExponentialHawkes.from_state(params, z, rng)
    simulator.time = s
    return counts(simulator, s)

In [ ]:
target = params.forward_mean_counts(z, h)
rng = np.random.default_rng(3)
fan_scores = {}
for estimate in (estimate_a, estimate_b, estimate_c, estimate_d):
    sample = np.array([estimate(rng) for _ in range(3000)])
    fan_scores[estimate.__name__] = scores(sample, target)
    print(f"{estimate.__name__}: mean {sample.mean(axis=0).round(3)} against {target.round(3)}, "
          f"scores {fan_scores[estimate.__name__].round(1)}")
right = {name: on(score) for name, score in fan_scores.items()}
wrong = {name: off(score) for name, score in fan_scores.items()}
assert all(right[name] or wrong[name] for name in fan_scores)
verify("question.fanStartsTheClockAtZero", "D", [
    right["estimate_a"] and wrong["estimate_b"] and wrong["estimate_c"] and wrong["estimate_d"],
    right["estimate_a"] and right["estimate_b"] and wrong["estimate_c"] and right["estimate_d"],
    right["estimate_a"] and wrong["estimate_b"] and right["estimate_c"] and right["estimate_d"],
    right["estimate_a"] and wrong["estimate_b"] and wrong["estimate_c"] and right["estimate_d"],
    all(right.values()),
])

## `question.forecastBetweenEventsFollowsTheFlow`: a forecast on a timer

At $t = 10$ s every path has had its last event at some $T_n \le t$, and the four forecasts
are scored on the mean, over 3000 paths, of the count realised in $(t, t + h]$ less the
forecast.

In [ ]:
params = config.one_way_pair_params()
t, h = 10.0, 1.0

def forecast_a(last, state):
    z = state * np.exp(-params.decay * (t - last))
    return params.forward_mean_counts(z, h)

def forecast_b(last, state):
    return params.forward_mean_counts(state, h)

def forecast_c(last, state):
    return (params.forward_mean_counts(state, t - last + h)
            - params.forward_mean_counts(state, t - last))

def forecast_d(last, state):
    z = state * np.exp(-params.decay * t)
    return params.forward_mean_counts(z, h)

In [ ]:
rng = np.random.default_rng(9)
forecasts = (forecast_a, forecast_b, forecast_c, forecast_d)
errors = {forecast: [] for forecast in forecasts}
for _ in range(3000):
    simulator = ExponentialHawkes(params, rng)
    last, state = 0.0, np.zeros(2)
    for time, _ in simulator.events(t):
        last, state = time, simulator.decayed_counts.copy()
    ahead = [event_type for _, event_type in simulator.events(t + h)]
    realised = np.bincount(ahead, minlength=2)
    for forecast in forecasts:
        errors[forecast].append(realised - forecast(last, state))

In [ ]:
timer_scores = {}
for forecast in forecasts:
    timer_scores[forecast.__name__] = scores(errors[forecast], np.zeros(2))
    print(f"{forecast.__name__}: mean error {np.mean(errors[forecast], axis=0).round(3)}, "
          f"scores {timer_scores[forecast.__name__].round(1)}")
right = {name: on(score) for name, score in timer_scores.items()}
wrong = {name: off(score) for name, score in timer_scores.items()}
assert all(right[name] or wrong[name] for name in timer_scores)
verify("question.forecastBetweenEventsFollowsTheFlow", "A", [
    right["forecast_a"] and wrong["forecast_b"] and wrong["forecast_c"] and wrong["forecast_d"],
    right["forecast_a"] and wrong["forecast_b"] and right["forecast_c"] and wrong["forecast_d"],
    wrong["forecast_a"] and wrong["forecast_b"] and right["forecast_c"] and wrong["forecast_d"],
    right["forecast_a"] and right["forecast_b"] and wrong["forecast_c"] and wrong["forecast_d"],
])

## `question.forecastAfterAMarketSellUsesThePostJumpState`: the state at $s$, and after it

Without the jump the forecast misses the descendants in $(s, s + h]$ of the sell at $s$,
$A\,\Phi(h)\,\mathbf e_2 = (0, 2.4\,(1 - e^{-1.6})/1.6)$, the second column of $K$ being
$-1.6\,\mathbf e_2$.

In [ ]:
params = config.one_way_pair_params()
h = 1.0

def forecast_error(times, types, k, post_jump):
    s = times[k]
    z = decayed_counts_at(params.decay, times, types, np.zeros(2),
                          Clock(times[k:k + 1]))[0]
    if post_jump:
        z = z + np.eye(2)[types[k]]
    ahead = (times > s) & (times <= s + h)
    realised = np.bincount(types[ahead], minlength=2)
    return realised - params.forward_mean_counts(z, h)

def path(rng, horizon):
    events = list(ExponentialHawkes(params, rng).events(horizon))
    times = np.array([time for time, _ in events])
    types = np.array([event_type for _, event_type in events])
    return times, types

In [ ]:
rng = np.random.default_rng(7)
errors = {False: [], True: []}
for _ in range(2000):
    times, types = path(rng, 30.0)
    k = np.flatnonzero(types == 1)[0]
    for post_jump in errors:
        error = forecast_error(times, types, k, post_jump)
        errors[post_jump].append(error)

Every path has its first sell before $30 - h$, so that the count over $(s, s + h]$ is
complete; the same seed draws the same paths again.

In [ ]:
rng = np.random.default_rng(7)
assert all(
    path_times[np.flatnonzero(path_types == 1)[0]] + h <= 30.0
    for path_times, path_types in (path(rng, 30.0) for _ in range(2000))
)
pre, post = scores(errors[False], np.zeros(2)), scores(errors[True], np.zeros(2))
print("mean errors from Z(s):  ", np.mean(errors[False], axis=0).round(3), "scores", pre.round(1))
print("mean errors from Z(s+): ", np.mean(errors[True], axis=0).round(3), "scores", post.round(1))
descendants_of_the_sell = params.excitation @ params.mean_response_integral(h) @ np.eye(2)[1]
print("A Phi(h) e_sell =", descendants_of_the_sell.round(3))
short_of_sells = on(scores(errors[False], descendants_of_the_sell)) and off(pre)
verify("question.forecastAfterAMarketSellUsesThePostJumpState", "B", [
    on(pre) and on(post),
    short_of_sells and on(post),
    bool(pre[0] > 8 and pre[1] > 8) and on(post),
    on(pre) and bool(post[1] < -8),
])

## `question.forecastFromAStoppingTime`: four rules for the time of the forecast

`forecast_error` and `path` are those of the previous question.

In [ ]:
def first_after_ten(times, types):
    return np.flatnonzero(times > 10.0)[0]

def first_after_a_lull(times, types):
    lull = np.diff(times, prepend=0.0) > 0.5
    return np.flatnonzero((times > 10.0) & lull)[0]

def first_before_a_lull(times, types):
    lull = np.diff(times, append=np.inf) > 0.5
    return np.flatnonzero((times > 10.0) & lull)[0]

def largest_state(times, types):
    zero = np.zeros(2)
    z = decayed_counts_at(params.decay, times, types, zero, Clock(times))
    total = (z + np.eye(2)[types]).sum(axis=1)
    return np.argmax(total[times < 39.0])

In [ ]:
rng = np.random.default_rng(8)
rules = (first_after_ten, first_after_a_lull,
         first_before_a_lull, largest_state)
errors = {rule: [] for rule in rules}
for _ in range(2000):
    times, types = path(rng, 40.0)
    for rule in rules:
        k = rule(times, types)
        errors[rule].append(forecast_error(times, types, k, True))

In [ ]:
rng = np.random.default_rng(8)
assert all(
    path_times[rule(path_times, path_types)] + h <= 40.0
    for path_times, path_types in (path(rng, 40.0) for _ in range(2000))
    for rule in rules
)
rule_scores = {}
for rule in rules:
    rule_scores[rule.__name__] = scores(errors[rule], np.zeros(2))
    print(f"{rule.__name__:>20}: mean error {np.mean(errors[rule], axis=0).round(3)}, "
          f"scores {rule_scores[rule.__name__].round(1)}")
right = {name: on(score) for name, score in rule_scores.items()}
wrong = {name: off(score) for name, score in rule_scores.items()}
assert all(right[name] or wrong[name] for name in rule_scores)
names = [rule.__name__ for rule in rules]
unbiased = {name for name in names if right[name]}
verify("question.forecastFromAStoppingTime", "E", [
    unbiased == set(names),
    unbiased == {"first_after_ten"},
    unbiased == {"first_after_ten", "first_after_a_lull", "first_before_a_lull"},
    unbiased == {"first_after_ten", "first_after_a_lull", "largest_state"},
    unbiased == {"first_after_ten", "first_after_a_lull"},
])

## `question.forecastAWindowThatOpensLater`: the counts of the second second

The four functions are compared with each other, and with a fan of 6000 paths continued
from $z$, read at the horizons 1 and 2.

In [ ]:
params = config.one_way_pair_params()
s, z = 30.0, np.array([0.5, 2.0])

def window_a(z):
    return (params.forward_mean_counts(z, 2.0)
            - params.forward_mean_counts(z, 1.0))

def window_b(z):
    m = params.forward_mean_state(z, 1.0)
    return params.forward_mean_counts(m, 1.0)

def window_c(z):
    return params.forward_mean_counts(z, 1.0)

def window_d(z):
    return (params.forward_mean_counts(z, s + 2.0)
            - params.forward_mean_counts(z, s + 1.0))

In [ ]:
fan = Fan.sample(params, z, 2.0, 6000, 11)
readings = ForwardCounts().read(fan, Clock(np.array([1.0, 2.0])))
second = readings[:, 1] - readings[:, 0]
window_scores = {}
for window in (window_a, window_b, window_c, window_d):
    window_scores[window.__name__] = scores(second, window(z))
    print(f"{window.__name__}: {window(z).round(4)}, scores of the fan {window_scores[window.__name__].round(1)}")
print("fan mean", second.mean(axis=0).round(4))
right = {name: on(score) for name, score in window_scores.items()}
wrong = {name: off(score) for name, score in window_scores.items()}
assert all(right[name] or wrong[name] for name in window_scores)
assert np.allclose(window_a(z), window_b(z))
gives = {name for name in window_scores if right[name]}
verify("question.forecastAWindowThatOpensLater", "B", [
    gives == {"window_a"},
    gives == {"window_a", "window_b"},
    gives == {"window_a", "window_b", "window_d"},
    gives == {"window_a", "window_d"},
    gives == {"window_a", "window_b", "window_c", "window_d"},
])

## `question.liveStateByRecursionAgainstRecomputation`: the state at every event

Both functions return the post-jump states $Z(T_k+)$, checked first on 4000 events.
Each is timed on the first $n$ events of one long path of the one-way pair, for $n$ from
1000 to 16000, and the choices are read off the time per event.

In [ ]:
def live_states_a(decay, times, types):
    state, last, out = np.zeros(2), 0.0, []
    for now, event_type in zip(times, types):
        state = state * np.exp(-decay * (now - last))
        state[event_type] += 1.0
        last = now
        out.append(state)
    return np.array(out)

def live_states_b(decay, times, types):
    out = []
    for k in range(len(times)):
        weights = np.exp(-decay * (times[k] - times[: k + 1]))
        out.append(np.bincount(types[: k + 1], weights, minlength=2))
    return np.array(out)

In [ ]:
params = config.one_way_pair_params()
long_times, long_types = path(np.random.default_rng(2), 12000.0)
assert long_times.size >= 16_000
first_times, first_types = long_times[:4000], long_types[:4000]
after_jump = decayed_counts_at(
    params.decay, first_times, first_types, np.zeros(2), Clock(first_times)
) + np.eye(2)[first_types]
assert np.allclose(live_states_a(params.decay, first_times, first_types), after_jump)
assert np.allclose(live_states_b(params.decay, first_times, first_types), after_jump)


def seconds_per_event(live_states, n):
    times, types = long_times[:n], long_types[:n]
    best = min(timeit.repeat(lambda: live_states(params.decay, times, types), number=1, repeat=3))
    return best / n


timings = {}
for n in (1000, 2000, 4000, 8000, 16000):
    timings[n] = (seconds_per_event(live_states_a, n), seconds_per_event(live_states_b, n))
    print(f"n = {n:>6}:  A {timings[n][0] * 1e6:6.2f} us   B {timings[n][1] * 1e6:7.2f} us per event")

growth_a = timings[16000][0] / timings[1000][0]
growth_b = timings[16000][1] / timings[1000][1]
b_over_a = timings[16000][1] / timings[16000][0]
b_over_a_small = timings[1000][1] / timings[1000][0]
print(f"growth per event from 1000 to 16000: A {growth_a:.1f}x, B {growth_b:.1f}x; "
      f"B over A: {b_over_a_small:.1f}x at 1000, {b_over_a:.1f}x at 16000")
verify("question.liveStateByRecursionAgainstRecomputation", "D", [
    b_over_a_small < 1 and b_over_a < 1,
    growth_a < 1.5 and growth_b < 1.5 and b_over_a > 1,
    growth_a > 3 and growth_b > 3,
    growth_a < 1.5 and growth_b > 3 and b_over_a > 10,
])